In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('person_month.csv.gz', compression='gzip')

In [ ]:
cols = [
    # Человек и время
    "SSUID", "PNUM", "reference_year", "MONTHCODE", "SPANEL", "SWAVE",

    # Штат проживания и переезды
    "TEHC_ST", "TST_INTV", "TMOVER",

    # Доходы, занятость и часы
    "TPEARN", "TPEARN_ALT", "RMESR", "TMWKHRS",

    # Пол, месячный возраст, раса, происхождение, образование и семейное положение
    "ESEX", "TAGE_EHC", "ERACE", "EORIGIN", "EEDUC", "EMS_EHC",

    # Совместное проживание, родители, супруг/партнер, размер домохозяйства и дети в домохозяйстве
    "ERESIDENCEID",
    "EPNSPOUS_EHC", 
    "EPNCOHAB_EHC",
    "RHNUMPER", "RHNUMU18",

    # Флаги качества и участия
    "APEARN", "APEARN_ALT", "AMESR", "AMWKHRS",
    "ASEX", "AEHC_ST",
    "RIN_UNIV", "EINTTYPE", "THHLDSTATUS",

    # Вес и параметры дизайна обследования — сохраняем на будущее
    "WPFINWGT", "GVARSTR", "GHLFSAM",

    # Происхождение записи
    # "release_year", "source_file",
]

df_research = df.loc[:, cols].copy()
df_research 

,SSUID,PNUM,reference_year,MONTHCODE,SPANEL,SWAVE,TEHC_ST,TST_INTV,TMOVER,TPEARN,...,AMESR,AMWKHRS,ASEX,AEHC_ST,RIN_UNIV,EINTTYPE,THHLDSTATUS,WPFINWGT,GVARSTR,GHLFSAM
0,11428577022,101,2021,1,2022,1,20,20.0,NaN,4131.0,...,9,9,1,1,1,1,2,8896.378211,31,1
1,11428577022,101,2021,2,2022,1,20,20.0,1.0,3612.0,...,9,9,1,1,1,1,2,8743.061762,31,1
2,11428577022,101,2021,3,2022,1,20,20.0,1.0,3470.0,...,9,9,1,1,1,1,2,9043.811867,31,1
3,11428577022,101,2021,4,2022,1,20,20.0,1.0,3523.0,...,9,9,1,1,1,1,2,9070.065262,31,1
4,11428577022,101,2021,5,2022,1,20,20.0,1.0,3790.0,...,9,9,1,1,1,1,2,8922.755157,31,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714991,88199599599922,201,2024,8,2022,4,6,6.0,1.0,NaN,...,9,0,1,1,1,1,1,6391.884880,142,2
714992,88199599599922,201,2024,9,2022,4,6,6.0,1.0,NaN,...,9,0,1,1,1,1,1,6200.257534,142,2
714993,88199599599922,201,2024,10,2022,4,6,6.0,1.0,NaN,...,9,0,1,1,1,1,1,6131.621860,142,2
714994,88199599599922,201,2024,11,2022,4,6,6.0,1.0,NaN,...,9,0,1,1,1,1,1,6118.780525,142,2


In [ ]:
import sys

sys.path.insert(0, "/Users/macbook/Documents/Codex/2026-11-08/z-x20/outputs")

from add_restriction_flag import add_restriction_flag
df_research = add_restriction_flag(df_research, policy_dir=("/Users/macbook/Documents/Codex/2026-11-08/z-x20/" "outputs/sipp_dobbs_2021_2024/policy"),)

df_research[[
    "TEHC_ST", "reference_year", "MONTHCODE",
    "restriction_flag", 
    "restriction_month_end",
    "restriction_share", "policy_uncertain",
]].head()

,TEHC_ST,reference_year,MONTHCODE,restriction_flag
0,20,2021,1,0
1,20,2021,2,0
2,20,2021,3,0
3,20,2021,4,0
4,20,2021,5,0


In [8]:
df_research.to_csv('abortion_research.csv.gz', index=False, compression='gzip')

# 2 итерация

In [2]:
import pandas as pd
df_research = pd.read_csv('abortion_research.csv.gz')

# Понятное название показателя дохода
df_research["TPEARN"] = pd.to_numeric(df_research["TPEARN"], errors="coerce").replace([-100000998, float("inf"), -float("inf")],float("nan"),)
df_research["TPEARN_ALT"] = pd.to_numeric(df_research["TPEARN_ALT"], errors="coerce").replace([-100000998, float("inf"), -float("inf")],float("nan"),)
df_research["date"] = pd.to_datetime(
    {"year": pd.to_numeric(df_research["reference_year"], errors="coerce"),
      "month": pd.to_numeric(df_research["MONTHCODE"], errors="coerce"),
      "day": 1,
    }, errors="coerce")


# df_research = df_research.rename(columns={"TPEARN": "income"})

# Название штата становится столбцом
df_research["state"] = pd.to_numeric(
    df_research["TEHC_ST"], errors="coerce").map({
    1: "Alabama", 2: "Alaska", 4: "Arizona", 5: "Arkansas",
    6: "California", 8: "Colorado", 9: "Connecticut",
    10: "Delaware", 11: "District of Columbia", 12: "Florida",
    13: "Georgia", 15: "Hawaii", 16: "Idaho", 17: "Illinois",
    18: "Indiana", 19: "Iowa", 20: "Kansas", 21: "Kentucky",
    22: "Louisiana", 23: "Maine", 24: "Maryland",
    25: "Massachusetts", 26: "Michigan", 27: "Minnesota",
    28: "Mississippi", 29: "Missouri", 30: "Montana",
    31: "Nebraska", 32: "Nevada", 33: "New Hampshire",
    34: "New Jersey", 35: "New Mexico", 36: "New York",
    37: "North Carolina", 38: "North Dakota", 39: "Ohio",
    40: "Oklahoma", 41: "Oregon", 42: "Pennsylvania",
    44: "Rhode Island", 45: "South Carolina", 46: "South Dakota",
    47: "Tennessee", 48: "Texas", 49: "Utah", 50: "Vermont",
    51: "Virginia", 53: "Washington", 54: "West Virginia",
    55: "Wisconsin", 56: "Wyoming",
})

df_research["ESEX"] = pd.to_numeric(df_research["ESEX"], errors="coerce")
df_research["restriction_flag"] = pd.to_numeric(df_research["restriction_month_end"], errors="coerce").astype('Int8')

In [3]:
df_research["female_flag"] = (
    pd.to_numeric(df_research["ESEX"], errors="coerce")
    .map({1: 0, 2: 1})
    .astype("Int8")
)

In [4]:
df_research = df_research.rename(columns={
    "TAGE_EHC": "age",
    "TMWKHRS": "weekly_hours",
    'TPEARN': 'income',
    "TPEARN_ALT": "income_alt",

    "RMESR": "employment_status_code",
    "EEDUC": "education_code",
    "ERACE": "race_code",
    "EMS_EHC": "marital_status_code",

    "RHNUMPER": "household_size",
    "RHNUMU18": "household_children_u18",

    "WPFINWGT": "survey_weight",
})

In [5]:
ssuid = (
    df_research["SSUID"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
    .str.zfill(14)
)

pnum = (
    df_research["PNUM"]
    .astype("string")
    .str.replace(r"\.0$", "", regex=True)
)

df_research["person_id"] = ssuid + "_" + pnum

In [6]:
df_research.columns

Index(['SSUID', 'PNUM', 'reference_year', 'MONTHCODE', 'SPANEL', 'SWAVE',
       'TEHC_ST', 'TST_INTV', 'TMOVER', 'income', 'income_alt',
       'employment_status_code', 'weekly_hours', 'ESEX', 'age', 'race_code',
       'EORIGIN', 'education_code', 'marital_status_code', 'ERESIDENCEID',
       'RPNPAR1_EHC', 'RPNPAR2_EHC', 'RPAR1TYP_EHC', 'RPAR2TYP_EHC',
       'EPNSPOUS_EHC', 'EPNCOHAB_EHC', 'household_size',
       'household_children_u18', 'APEARN', 'APEARN_ALT', 'AMESR', 'AMWKHRS',
       'ASEX', 'AEHC_ST', 'RIN_UNIV', 'EINTTYPE', 'THHLDSTATUS',
       'survey_weight', 'GVARSTR', 'GHLFSAM', 'restriction_flag',
       'restriction_month_end', 'restriction_share',
       'restriction_days_confirmed', 'restriction_days_uncertain',
       'policy_uncertain', 'policy_covered', 'date', 'state', 'female_flag',
       'person_id'],
      dtype='str')

In [7]:
main_columns = [
    "person_id", "date", "state",
    "female_flag", "age",
    "income", "income_alt", "weekly_hours", "employment_status_code",
    "education_code", "race_code", "marital_status_code",
    "household_size", "household_children_u18",
    "restriction_flag",  "restriction_share", #"policy_uncertain",
]


df_research_short = df_research[main_columns].head()

In [8]:
# Полная версия для дополнительных проверок
df_research_full = df_research.copy()

columns = [
    # 1. Человек, время и география
    "person_id",
    "date",
    "state",
    "TEHC_ST",

    # 2. Пол и социально-демографические характеристики
    "female_flag",
    "age",
    "race_code",
    "EORIGIN",
    "education_code",
    "marital_status_code",

    # 3. Доходы и рынок труда
    "income",
    "income_alt",
    "employment_status_code",
    "weekly_hours",

    # 4. Ограничения в штате
    "restriction_flag",
    "restriction_share",

    # 5. Домохозяйство и связи с родителями
    "household_size",
    "household_children_u18",
    # "ERESIDENCEID",
    # "RPNPAR1_EHC",
    # "RPNPAR2_EHC",
    # "RPAR1TYP_EHC",
    # "RPAR2TYP_EHC",

    # 6. Качество основных показателей и участие
    # "APEARN",
    # "AMESR",
    # "AMWKHRS",
    # "RIN_UNIV",
    # "EINTTYPE",

    # 7. Вес
    "survey_weight",
]

# В текущем датасете эти два флага должны совпадать
assert (
    df_research_full["restriction_flag"].fillna(-1).eq(df_research_full["restriction_month_end"].fillna(-1)).all()
), "Флаги различаются — restriction_month_end пока нужно сохранить"

df_research = df_research_full.loc[:, columns].copy()

print(df_research.shape)
df_research.head()

(714996, 19)


,person_id,date,state,TEHC_ST,female_flag,age,race_code,EORIGIN,education_code,marital_status_code,income,income_alt,employment_status_code,weekly_hours,restriction_flag,restriction_share,household_size,household_children_u18,survey_weight
0,00011428577022_101,2021-01-01,Kansas,20,1,31,2,2,42.0,1.0,4131.0,4067.0,1.0,41.5,0,0.0,4,2,8896.378211
1,00011428577022_101,2021-02-01,Kansas,20,1,31,2,2,42.0,1.0,3612.0,3880.0,1.0,42.0,0,0.0,4,2,8743.061762
2,00011428577022_101,2021-03-01,Kansas,20,1,31,2,2,42.0,1.0,3470.0,3405.0,1.0,42.0,0,0.0,4,2,9043.811867
3,00011428577022_101,2021-04-01,Kansas,20,1,31,2,2,42.0,1.0,3523.0,3572.0,1.0,42.0,0,0.0,4,2,9070.065262
4,00011428577022_101,2021-05-01,Kansas,20,1,31,2,2,42.0,1.0,3790.0,3722.0,1.0,42.0,0,0.0,4,2,8922.755157


In [10]:
df_research.to_csv('../abortion_research_data.csv.gz', index=False, compression='gzip')